In [13]:
from bs4 import BeautifulSoup
from datetime import datetime
import requests
import json

In [ ]:
# Webhook URL
WEBHOOK_URL = "Webhook"

# API KEY
API_KEY = "Key"

In [15]:
# Determine date today
date_str = datetime.now().strftime("%Y-%m-%d")

# Scrape the biggest news sites today

In [16]:
# Note: Commented ones that cannot be scraped due to robot.txt

news_sites = [
    # ============ GLOBAL AGGREGATORS ============
    "https://www.msn.com",
    "https://news.google.com",
    "https://news.yahoo.com",
    "https://ca.news.yahoo.com",
    "https://finance.yahoo.com",
    "https://www.newsnow.co.uk",
    # "https://substack.com",
    "https://news.ycombinator.com",
    # "https://www.allsides.com",
    "https://ground.news",

    # ============ WIRES / AGENCIES ============
    # "https://apnews.com",
    # "https://www.reuters.com",
    "https://www.afp.com",
    # "https://www.bloomberg.com",
    # "https://www.upi.com",
    "https://www.thecanadianpress.com",

    # ============ MAJOR INTERNATIONAL BRANDS ============
    "https://www.bbc.com",
    "https://www.theguardian.com",
    "https://www.nytimes.com",
    # "https://www.washingtonpost.com",
    # "https://www.wsj.com",
    # "https://www.ft.com",
    # "https://www.economist.com",
    "https://www.cnn.com",
    "https://www.aljazeera.com",
    "https://www.npr.org",
    "https://www.dw.com",
    # "https://www.france24.com",
    "https://www.euronews.com",
    # "https://www.scmp.com",

    # ============ CANADA — NATIONAL BROADCASTERS ============
    # "https://www.cbc.ca",
    "https://ici.radio-canada.ca",
    "https://www.ctvnews.ca",
    "https://globalnews.ca",
    "https://www.citynews.ca",
    "https://www.cp24.com",
    "https://www.aptnnews.ca",
    "https://www.bnnbloomberg.ca",
    "https://www.noovo.info",

    # ============ CANADA — NATIONAL / BUSINESS ============
    "https://www.theglobeandmail.com",
    # "https://nationalpost.com",
    "https://www.thestar.com",
    # "https://financialpost.com",
    # "https://macleans.ca",
    # "https://thewalrus.ca",
    # "https://www.thelogic.co",
    "https://betakit.com",
    # "https://thehub.ca",
    # "https://www.hilltimes.com",
    # "https://www.ipolitics.ca",

    # ============ CANADA — POSTMEDIA / SUN CHAIN ============
    # "https://torontosun.com",
    # "https://ottawacitizen.com",
    # "https://ottawasun.com",
    "https://montrealgazette.com",
    # "https://vancouversun.com",
    # "https://theprovince.com",
    # "https://calgaryherald.com",
    # "https://calgarysun.com",
    # "https://edmontonjournal.com",
    # "https://edmontonsun.com",
    "https://winnipegsun.com",
    # "https://windsorstar.com",
    # "https://lfpress.com",
    "https://thestarphoenix.com",
    "https://leaderpost.com",
    # "https://www.thewhig.com",

    # ============ CANADA — OTHER REGIONAL DAILIES ============
    "https://www.winnipegfreepress.com",
    "https://www.thespec.com",
    # "https://www.timescolonist.com",
    # "https://www.saltwire.com",
    # "https://www.thechronicleherald.ca",
    # "https://www.thetelegram.com",
    # "https://www.theguardian.pe.ca",
    # "https://tj.news",
    # "https://www.biv.com",

    # ============ CANADA — QUEBEC / FRENCH ============
    "https://www.lapresse.ca",
    "https://www.journaldemontreal.com",
    "https://www.journaldequebec.com",
    "https://www.ledevoir.com",
    "https://www.tvanouvelles.ca",
    "https://www.lesoleil.com",
    "https://www.ledroit.com",
    "https://www.latribune.ca",
    "https://www.lequotidien.com",
    # "https://www.lanouvelliste.ca",

    # ============ CANADA — DIGITAL-NATIVE / CITY ============
    "https://dailyhive.com",
    "https://www.blogto.com",
    "https://www.narcity.com",
    # "https://nowtoronto.com",
    # "https://www.sootoday.com",
    # "https://www.barrietoday.com",
    # "https://www.coastreporter.net",

    # ============ CANADA — INDEPENDENT / OPINION ============
    "https://thetyee.ca",
    # "https://www.nationalobserver.com",
    # "https://thenarwhal.ca",
    "https://www.canadaland.com",
    # "https://rabble.ca",
    "https://thebreach.ca",
    # "https://www.thepostmillennial.com",
    # "https://www.westernstandard.news",
    # "https://www.rebelnews.com",
    # "https://tnc.news",
]

In [17]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class WebScraper:
    def __init__(self, delay_range=(1, 3), user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'):
        self.delay_range = delay_range
        self.session = requests.Session()
        self.session.headers.update({'User-Agent': user_agent})
        
    def can_scrape(self, url):
        """Check if we can scrape a URL using robots.txt"""
        parsed_url = urlparse(url)
        robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"
        
        rp = RobotFileParser()
        rp.set_url(robots_url)
        
        try:
            rp.read()
            return rp.can_fetch(self.session.headers['User-Agent'], url)
        except Exception as e:
            logger.warning(f"Could not check robots.txt for {url}: {e}")
            return True  # Assume we can scrape if robots.txt check fails
    
    def scrape_url(self, url, parse_method='soup'):
        """Scrape a single URL with error handling"""
        if not self.can_scrape(url):
            logger.warning(f"Cannot scrape {url} according to robots.txt")
            return None
        
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            
            if parse_method == 'soup':
                return BeautifulSoup(response.content, 'html.parser')
            elif parse_method == 'text':
                return response.text
            else:
                return response
                
        except requests.exceptions.RequestException as e:
            logger.error(f"Error scraping {url}: {e}")
            return None
        
        finally:
            # Random delay to be respectful
            time.sleep(random.uniform(*self.delay_range))
    
    def scrape_multiple(self, urls, extract_method='get_text'):
        """Scrape multiple URLs and extract content"""
        results = {}
        
        for i, url in enumerate(urls):
            logger.info(f"Scraping ({i+1}/{len(urls)}): {url}")
            
            soup = self.scrape_url(url)
            if soup:
                if extract_method == 'get_text':
                    results[url] = soup.get_text(strip=True)
                elif extract_method == 'links':
                    results[url] = [a.get('href') for a in soup.find_all('a', href=True)]
                elif extract_method == 'titles':
                    results[url] = [title.get_text(strip=True) for title in soup.find_all(['h1', 'h2', 'h3'])]
                elif extract_method == 'all':
                    results[url] = {
                        'text': soup.get_text(strip=True),
                        'links': [a.get('href') for a in soup.find_all('a', href=True)],
                        'titles': [title.get_text(strip=True) for title in soup.find_all(['h1', 'h2', 'h3'])]
                    }
            else:
                results[url] = None
        
        return results


# Initialize scraper with random delays
scraper = WebScraper(delay_range=(1, 3))

# Scrape all URLs
results = scraper.scrape_multiple(news_sites, extract_method='all')

failed_scrapes = []

# Process results
for url, content in results.items():
    if content:
        print(f"\n=== {url} ===")
        if isinstance(content, dict):
            print(f"Text preview: {content['text'][:200]}...")
            print(f"Links found: {len(content['links'])}")
            print(f"Titles found: {len(content['titles'])}")
        else:
            print(f"Content: {content[:200]}...")
    else:
        print(f"Failed to scrape: {url}")
        failed_scrapes.append(url)


2026-08-24 14:22:49,462 - INFO - Scraping (1/50): https://www.msn.com
2026-08-24 14:22:51,520 - INFO - Scraping (2/50): https://news.google.com
2026-08-24 14:22:51,762 - WARNING - Cannot scrape https://news.google.com according to robots.txt
2026-08-24 14:22:51,762 - INFO - Scraping (3/50): https://news.yahoo.com
2026-08-24 14:22:54,931 - INFO - Scraping (4/50): https://ca.news.yahoo.com
2026-08-24 14:22:58,588 - INFO - Scraping (5/50): https://finance.yahoo.com
2026-08-24 14:23:01,834 - INFO - Scraping (6/50): https://www.newsnow.co.uk
2026-08-24 14:23:05,737 - INFO - Scraping (7/50): https://news.ycombinator.com
2026-08-24 14:23:09,422 - INFO - Scraping (8/50): https://ground.news
2026-08-24 14:23:12,402 - INFO - Scraping (9/50): https://www.afp.com
2026-08-24 14:23:15,390 - INFO - Scraping (10/50): https://www.thecanadianpress.com
2026-08-24 14:23:18,759 - INFO - Scraping (11/50): https://www.bbc.com
2026-08-24 14:23:21,262 - INFO - Scraping (12/50): https://www.theguardian.com
2026


=== https://www.msn.com ===
Text preview: MSN...
Links found: 0
Titles found: 0
Failed to scrape: https://news.google.com

=== https://news.yahoo.com ===
Text preview: Yahoo News: Latest and Breaking News, Headlines, Live Updates, and MoreSearch Query for Search the webSearch with Yahoo ScoutSearch the webNewsFinanceSportsMoreMailSearch the webNewsUSPoliticsScienceW...
Links found: 102
Titles found: 63

=== https://ca.news.yahoo.com ===
Text preview: News & Analysis | Yahoo News CanadaSearch Query for Search the webSearch the webNewsFinanceMoreMailSearch the webNewsCanadaWorld CupPoliticsWorldU.S.WeatherScience & TechPollsMade in CanadaGaza Confli...
Links found: 107
Titles found: 63

=== https://finance.yahoo.com ===
Text preview: Yahoo Finance - Stock Market Live, Quotes, Business & Finance NewsOops, something went wrongSkip to navigationSkip to main contentSkip to right columnYahoo FinanceNewsFinanceSportsMoreNewsToday's news...
Links found: 338
Titles found: 37

=== https://www.ne

# Find the biggest news in the world today

In [ ]:
# Ask OpenRouter for the top news in the world today.

# Code below found on their website: https://openrouter.ai/openrouter/free
prompt = (
    f"Today is {date_str}\n\n"
    "Summarize the top 10 most important news today. \n\n"
    "Mention the sources for each news on your final response.\n\n"
    "Final response should be concise and under 2,000 characters."
    f"{json.dumps(results, indent=2)}"
)

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer " + API_KEY,
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "openrouter/free",
    "messages": [
        {
          "role": "user",
          "content": prompt
        }
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']

In [19]:
response

{'role': 'assistant',
 'content': "Most important news today (2026-08-24):\n\n1. Trump threatens 50% tariffs on Canadian autos and steel - Canada calls it economic war\n2. Carney unveils $11 billion for 6 icebreakers in Quebec amid trade war\n3. Wildfire forces 90,000 to evacuate in Nevada\n4. Theresa Harp deleted tweets about Jan 6 come to light\n5. Ontario Premier Ford threatens to cut electricity to US\n6. US Senate trade talks collapse with Canada\n7. Mark Carney says Canada 'no longer subsidiary' of US\n8. Maria Harp's deleted tweets about Jan 6 revealed\n9. Donald Trump threatens to double auto tariffs\n10. Empty hospitals in British Columbia\n11. Two former B.C. Conservatives plan new party\n12. Terrorist attack on synagogue in Quebec\n13. Railway worker arrested at border\n13. Carney says Canada won't be rush to sign trade deal\n14. Quebec celebrates 150th anniversary of Treaty 6\n15. Cold front brings record temperatures in BC\n16. Group celebrates 150th anniversary of Treaty 

# Post on Webhook (Discord)

In [ ]:
# If some requests failed to scrape, inform user
if failed_scrapes != []:
    requests.post(
        WEBHOOK_URL,
        json={"content": "@everyone failed scrapes for: "
              f"{failed_scrapes}"}
    )

raw = response.get('content')
if isinstance(raw, str):
    limited_content = raw[:2000]
else:
    limited_content = str(raw)[:2000]  # convert to string first, then slice

requests.post(
    WEBHOOK_URL,
    json={"content": limited_content}
)

<Response [204]>